# **Project: Predicting 30-Day Readmission for Diabetic Patients**

# **1. Business Understanding**

The dataset contains ten years (1999–2008) of clinical data from 130 U.S. hospitals, focusing on patients diagnosed with diabetes. The main business problem is the high rate of 30-day readmissions among diabetic patients, which leads to increased hospital costs, penalties, and poorer patient outcomes due to complications from inadequate glycemic control.  
The business objective is to build a predictive model that identifies patients at high risk of early readmission before discharge. This would allow hospitals to improve diabetes management, reduce preventable readmissions, and enhance overall quality of care.


# **2. Data Understanding**

The dataset consists of hospital records including demographics, admission details, diagnosis codes, medications, lab results, and prior hospital visits. The target variable indicates whether the patient was readmitted within 30 days.  
Initial exploration shows a mix of categorical and numeric features, missing values represented by `"?"`, class imbalance in the readmission outcome, and several high-cardinality attributes. These characteristics suggest the need for careful preprocessing before modeling, particularly for machine learning methods such as RLT.

# **3. Data Preparation**


In [37]:

import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

In [24]:
# 1. Load the dataset (adjust path as needed)
df = pd.read_csv(r"C:\Users\LENOVO\Downloads\diabetic_data.csv")

# Quick look
print("Initial shape:", df.shape)
print(df.head())
print(df.isnull().sum())

Initial shape: (101766, 50)
   encounter_id  patient_nbr             race  gender      age weight  \
0       2278392      8222157        Caucasian  Female   [0-10)      ?   
1        149190     55629189        Caucasian  Female  [10-20)      ?   
2         64410     86047875  AfricanAmerican  Female  [20-30)      ?   
3        500364     82442376        Caucasian    Male  [30-40)      ?   
4         16680     42519267        Caucasian    Male  [40-50)      ?   

   admission_type_id  discharge_disposition_id  admission_source_id  \
0                  6                        25                    1   
1                  1                         1                    7   
2                  1                         1                    7   
3                  1                         1                    7   
4                  1                         1                    7   

   time_in_hospital  ... citoglipton insulin  glyburide-metformin  \
0                 1  ...          No 

In [29]:
# 2) Replace '?' with NaN
# ------------------------------------------------
df = df.replace("?", np.nan)

# ------------------------------------------------
# 3) Drop identifier columns (not useful for ML)
# ------------------------------------------------
df = df.drop(["encounter_id", "patient_nbr"], axis=1)

# ------------------------------------------------
# 4) Drop extremely sparse column
#    max_glu_serum has ~90% missing → remove
# ------------------------------------------------
df = df.drop("max_glu_serum", axis=1)


In [31]:
# 5) Create binary target variable
#    1  = readmitted <30 days
#    0  = NO or >30
# ------------------------------------------------
df["readmitted_30"] = (df["readmitted"] == "<30").astype(int)
df = df.drop("readmitted", axis=1)

In [33]:
# 6) Separate categorical and numerical columns
# ------------------------------------------------
categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()
numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()

# Remove target from numerical list if present
if "readmitted_30" in numerical_cols:
    numerical_cols.remove("readmitted_30")

print("Categorical columns:", len(categorical_cols))
print("Numerical columns:", len(numerical_cols))


Categorical columns: 35
Numerical columns: 11


In [39]:
# 7) Preprocessing pipeline
# ------------------------------------------------
preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), numerical_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", drop="first"), categorical_cols)
    ]
)

# Fit & transform the data
X_processed = preprocessor.fit_transform(df)

# ------------------------------------------------
# 8) Retrieve encoded column names
# ------------------------------------------------
cat_encoded_names = preprocessor.named_transformers_["cat"].get_feature_names_out(categorical_cols)
all_feature_names = numerical_cols + list(cat_encoded_names)

In [41]:
# 9) Build final cleaned DataFrame
# ------------------------------------------------
df_prepared = pd.DataFrame(
    X_processed.toarray() if hasattr(X_processed, "toarray") else X_processed,
    columns=all_feature_names
)

# Add target variable
df_prepared["readmitted_30"] = df["readmitted_30"].values

# ------------------------------------------------
# 10) Final check
# ------------------------------------------------
print("Final prepared dataset shape:", df_prepared.shape)
df_prepared.head()

Final prepared dataset shape: (101766, 2432)


,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,...,glyburide-metformin_No,glyburide-metformin_Steady,glyburide-metformin_Up,glipizide-metformin_Steady,glimepiride-pioglitazone_Steady,metformin-rosiglitazone_Steady,metformin-pioglitazone_Steady,change_No,diabetesMed_Yes,readmitted_30
0,6.0,25.0,1.0,1.0,41.0,0.0,1.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0
1,1.0,1.0,7.0,3.0,59.0,0.0,18.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0
2,1.0,1.0,7.0,2.0,11.0,5.0,13.0,2.0,0.0,1.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0
3,1.0,1.0,7.0,2.0,44.0,1.0,16.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0
4,1.0,1.0,7.0,1.0,51.0,0.0,8.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0


In [43]:
print(df_prepared.head())

   admission_type_id  discharge_disposition_id  admission_source_id  \
0                6.0                      25.0                  1.0   
1                1.0                       1.0                  7.0   
2                1.0                       1.0                  7.0   
3                1.0                       1.0                  7.0   
4                1.0                       1.0                  7.0   

   time_in_hospital  num_lab_procedures  num_procedures  num_medications  \
0               1.0                41.0             0.0              1.0   
1               3.0                59.0             0.0             18.0   
2               2.0                11.0             5.0             13.0   
3               2.0                44.0             1.0             16.0   
4               1.0                51.0             0.0              8.0   

   number_outpatient  number_emergency  number_inpatient  ...  \
0                0.0               0.0             